# 02 — Diagnostics: is this a siting model or an urbanization detector?

Before fitting anything, one question decides whether this project is worth
building: **do data centers land where Merascope's physical indicators say they
should, or do they simply land where cities are?**

Both data centers and OSM mapping effort concentrate in urban, industrial places.
A model trained naively will learn "cities", score well, and tell us nothing about
siting. This notebook is the check.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
import dcmodel as D

df = pd.read_csv(D.CACHE / "model_table.csv", dtype={"zcta": str})
y  = df["has_dc_any"]
print(f"{len(df):,} ZCTAs, {y.sum()} with a mapped data center ({100*y.mean():.2f}%)")

## 1. Univariate separation

Rank-based AUC for each indicator on its own. 0.5 = no signal. Below 0.5 means the
indicator runs *backwards* — Merascope scores these so that higher = more suitable,
so an AUC well under 0.5 means data centers prefer the "unsuitable" end.

In [ ]:
rows = []
for c in D.NAT_COLS + ["area_km2"]:
    v = pd.to_numeric(df[c], errors="coerce")
    r = rankdata(v.fillna(v.median()))
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    auc = (r[y == 1].sum() - n1*(n1+1)/2) / (n1*n0)
    rows.append({"feature": c, "mean_dc": v[y==1].mean(), "mean_nodc": v[y==0].mean(),
                 "auc": auc, "separation": abs(auc - 0.5)})

uni = pd.DataFrame(rows).sort_values("separation", ascending=False)
print(uni.head(16).to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

In [ ]:
top = uni.head(14).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#c44e52" if a < 0.5 else "#4c72b0" for a in top["auc"]]
ax.barh(top["feature"], top["auc"] - 0.5, left=0.5, color=colors)
ax.axvline(0.5, color="black", linewidth=1)
ax.set_xlabel("univariate AUC  (0.5 = no signal)")
ax.set_title("Blue: DCs prefer the high-score end.  Red: DCs prefer the low-score end.")
plt.tight_layout(); plt.show()

Read the top of that table carefully. The strongest separators are
`pop_exposure_score` (inverted — dense ZCTAs), `rcra_score` and
`contamination_score` (inverted — hazardous-waste and TRI sites, i.e. industrial
land), `ej_score` (inverted — higher community burden), and
`air_quality_score` (inverted — NAAQS nonattainment areas).

Every one of those is a proxy for *"this is an industrial part of a city."*

The physical siting indicators Merascope is actually built around — soil, aquifer,
seismic, flood, geothermal, water stress, slope — sit near 0.5.

## 2. The fiber overlap — how much of it is circular?

`fiber_score` and `fac_dist_m` are built from PeeringDB colocation facilities.
Colocation facilities *are* data centers, so some of this feature is the label
measured a second way. The question is how much.

In [ ]:
import requests, geopandas as gpd
from scipy.spatial import cKDTree

if not (D.CACHE / "peeringdb_fac.csv").exists():
    r = requests.get("https://www.peeringdb.com/api/fac?depth=0", timeout=180,
                     headers={"User-Agent": "merascope-research/1.0"})
    r.raise_for_status()
    f = pd.DataFrame(r.json()["data"])
    f = f[(f["country"] == "US") & f["latitude"].notna() & f["longitude"].notna()]
    f[["id","name","latitude","longitude","city","state"]].to_csv(
        D.CACHE / "peeringdb_fac.csv", index=False)

geom = gpd.read_file(D.CACHE / "zcta_geom.geojson")
dc   = D.load_datacenters()
df, dist = D.attach_colocation(df, geom, dc)

print("Distance from each OSM data center to the nearest PeeringDB facility:")
for t, lab in [(100, "100 m (same building)"), (1000, "1 km"), (5000, "5 km")]:
    print(f"  within {lab:<22} {(dist < t).sum():>5} / {len(dist)}  ({100*(dist < t).mean():.1f}%)")
print()
print(f"positives, all data centers : {df['has_dc_any'].sum()}")
print(f"positives, non-colocated    : {df['has_dc_noncolo'].sum()}")

So roughly a fifth of our labelled data centers are literally the same building as
a PeeringDB facility, and 43% are within a kilometre. That is real overlap, but it
is not the whole feature.

The decisive test is in notebook 03: rerun the model against `has_dc_noncolo`,
which keeps only positives at least 1 km from any PeeringDB facility. Against the
full label, fiber triples the lift (4.88x -> 14.97x). Against the non-colocated
label it still adds +33% (5.63x -> 7.48x).

**Conclusion: fiber_score is about two-thirds identity leakage and one-third real
signal.** Do not drop it, and do not quote the headline number that includes it —
report it against `has_dc_noncolo`, where its contribution is honest.

## 3. Spatial clustering of positives

Positives are not scattered — they cluster hard (Ashburn, Santa Clara, Dallas,
Phoenix). That means a random train/test split leaks: a neighbouring ZCTA of a
training positive lands in test and the model looks far better than it is.
Everything downstream uses spatially-blocked CV instead.

In [ ]:
print("Top 12 states by mapped-DC ZCTA count:")
print(df[df.has_dc_any == 1]["state"].value_counts().head(12).to_string())

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.scatter(df.loc[y==0, "lon"], df.loc[y==0, "lat"], s=0.5, c="#d8d8d8", linewidths=0)
ax.scatter(df.loc[y==1, "lon"], df.loc[y==1, "lat"], s=14, c="#c44e52", linewidths=0)
ax.set_xlim(-125, -66); ax.set_ylim(24, 50); ax.set_aspect(1.3)
ax.set_title("ZCTAs with a mapped data center (red)")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()